In [125]:
import pygame as pg # usei para montar a interface
import random # usei para determinar o comportamento de bots
import sys # usei para não dar erro quando fechar


pg.init()

largura = 700
altura = 600

tela = pg.display.set_mode((largura, altura))
pg.display.set_caption("JOGO DO DILEMA DOS PRISIONEIROS")


arial = pg.font.SysFont("Arial", 30)
arial_pequeno = pg.font.SysFont("Arial", 22)
arial_huge = pg.font.SysFont("Arial", 50)

# =========================================================
# CORES
# =========================================================

branco = (230,230,230)
preto = (0,0,0)
cor_do_fundo = (30,30,30)

verdin = (40,147,47)
vermelhin = (150,3,3)
bege = (194,176,146)
cinza = (50,50,50)
azulzin = (50,150,250)

# =========================================================
# BOTÕES DE ESCOLHA, TABELA e BOTÕES DE ETAPA 
# =========================================================

botao_cooperar = pg.Rect(50, 280, 200, 50)
botao_trair = pg.Rect(50, 340, 200, 50)

tabela_escolhas = pg.Rect(30, 200, 570, 220)

etapa_nao_lembra = pg.Rect(60, 270, 250, 300)
bot_random = pg.Rect(83, 350, 200, 50)
bot_bom = pg.Rect(83, 420, 200, 50)
bot_mal = pg.Rect(83, 490, 200, 50)
etapa_lembra = pg.Rect(360, 270, 250, 300)
bot_vingativo = pg.Rect(383, 350, 200, 50)
bot_imitador = pg.Rect(383, 420, 200, 50)
bot_bravo = pg.Rect(383, 490, 200, 50)

# =========================================================
# COMPORTAMENTO DOS BOTS
# =========================================================

machine_choices = ["trair", "cooperar"]

def randomico(): # comportamento aleatório
    return random.choice(machine_choices)

def bonzinho(): # sempre coopera
    return "cooperar"

def malvado(): # sempre trai
    return "trair"

def vingativo(): # começa bonzinho, mas se você traí-lo, fica malvado
    global T
    if T == 0:
        return "cooperar"
    else: 
        return "trair"
    
def imitador(): # começa cooperando e depois só copia o que você jogou por útlimo
    global c
    global t
    if c == 0 and t == 0:
        return "cooperar"
    elif c == 0:
        return "trair"
    elif t == 0:
        return "cooperar"
     
def bravo():
    global c
    global t
    if c == 0 and t == 0:
        return "cooperar"
    elif c < 2:
        return "trair"
    else:
        return "cooperar"

# =========================================================
# LÓGICA DO JOGO
# =========================================================

def payoff(machine_choice, user_choice):
    payoff = 0
    if user_choice == "trair":
        if machine_choice == "trair":
            payoff -= 5
        else:
            payoff -= 1
    elif user_choice == "cooperar":
        if machine_choice == "trair":
            payoff -= 10
        else:
            payoff -= 2
    return payoff

def desenhar_escolhas(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):

    pg.draw.rect(tela, cinza, tabela_escolhas)

    texto_user_choice = arial.render("VOCÊ ESCOLHE:", True, branco)

    texto_cooperar = arial.render("COOPERAR", True, preto)
    onde_texto_cooperar = texto_cooperar.get_rect(center = botao_cooperar.center)

    texto_trair = arial.render("TRAIR", True, preto)
    onde_texto_trair = texto_trair.get_rect(center = botao_trair.center)

    tela.blit(texto_user_choice, (50,220))

    # BOTÃO COOPERAR --------------------------------------------------------------------
    cor_cooperar = bege
    # if user_choice == "cooperar": # parte antiga que decidi tirar 
    #     cor_cooperar = verdin
    if botao_cooperar.collidepoint(mouse_pos):
        cor_cooperar = verdin
    pg.draw.rect(tela, cor_cooperar, botao_cooperar)

    tela.blit(texto_cooperar, onde_texto_cooperar)

    # BOTÃO TRAIR -------------------------------------------------------------
    cor_trair = bege
    # if user_choice == "trair": # parte antiga que decidi tirar 
    #     cor_trair = vermelhin
    if botao_trair.collidepoint(mouse_pos):
        cor_trair = vermelhin
    pg.draw.rect(tela, cor_trair, botao_trair)

    tela.blit(texto_trair, onde_texto_trair)

    texto_machine_choice = arial.render("BOT ESCOLHE:", True, branco)


    tela.blit(texto_machine_choice, (340,220))
    if machine_choice:
        texto_bot = arial.render(
            machine_choice.upper(),
            True,
            branco
        )

        tela.blit(texto_bot, (340,290))

    if resultado == 0:
        texto_resultado = arial.render(
        f'Anos de cadeia: ____',
        True,
        branco
    )
    else:
        texto_resultado = arial.render(
            f"Anos de cadeia: {-resultado}",
            True,
            branco
    )
    tela.blit(texto_resultado, (340,350))
    
    texto_reset = arial_pequeno.render(
        "PRESSIONE R PARA REINICIAR",
        True,
        branco
    )

    tela.blit(texto_reset, (50,20))

    texto_pontos_totais = arial.render(
        f'PENA TOTAL: {-total_points}',
        True,
        branco
    )

    tela.blit(texto_pontos_totais,(410,520))

    texto_trair_total = arial.render(
        f'Você TRAIU: {T}',
        True,
        branco
    )
    texto_cooperar_total = arial.render(
        f'Você COOPEROU: {C}',
        True,
        branco
    )

    tela.blit(texto_cooperar_total, (50,500))
    tela.blit(texto_trair_total, (50,540))


def resetar():
    global pode_clicar
    pode_clicar = True
    global user_choice
    user_choice = None
    global machine_choice
    machine_choice = None
    global resultado
    resultado = 0
    global total_points
    total_points = 0
    global C
    C = 0
    global T
    T = 0
    global c
    c = 0
    global t
    t = 0
    global frame_atual
    frame_atual = 0
    global tempo_frame
    tempo_frame = 0
    global animacao_terminou
    animacao_terminou = False

    return (
        pode_clicar, user_choice, machine_choice, 
        resultado, total_points, C, T, c, t,
        frame_atual, tempo_frame, animacao_terminou)

# =========================================================
# FUNÇÕES DE DESENHO
# =========================================================

frame_1_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-1.png"
).convert_alpha()

frame_2_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-2.png"
).convert_alpha()

frame_3_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-3.png"
).convert_alpha()

frame_4_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-4.png"
).convert_alpha()

frame_5_cadeia = pg.image.load(
    "FINAL_PCD_IMAGENS/image-5.png"
).convert_alpha()

# mesmo tamanho para todos os frames
frame_1_cadeia = pg.transform.smoothscale(frame_1_cadeia,(1800,1800))
frame_2_cadeia = pg.transform.smoothscale(frame_2_cadeia,(1800,1800))
frame_3_cadeia = pg.transform.smoothscale(frame_3_cadeia,(1800,1800))
frame_4_cadeia = pg.transform.smoothscale(frame_4_cadeia,(1800,1800))
frame_5_cadeia = pg.transform.smoothscale(frame_5_cadeia,(1800,1800))

# lista dos frames
frames_cadeia = [
    frame_1_cadeia,
    frame_2_cadeia,
    frame_3_cadeia,
    frame_4_cadeia,
    frame_5_cadeia
]
# posição dos frames
onde_frames_cadeia = (
    (155-largura),
    (-7.5-altura)
)

# VARIÁVEIS DA ANIMAÇÃO

frame_atual = 0

tempo_frame = 0

velocidade_animacao = 120

animacao_terminou = False

def desenhar_menu():

    tela.fill(cor_do_fundo)
    texto_menu_titulo = arial_huge.render("MENU", True, branco)

    texto_menu_1 = arial.render(
        "> : AVANÇAR",
        True,
        branco
    )

    texto_menu_2 = arial.render(
        "< : VOLTAR",
        True,
        branco
    )

    texto_menu_3 = arial.render("ESPAÇO : FECHAR", True, branco)


    tela.blit(texto_menu_titulo, (290,100))
    tela.blit(texto_menu_1, (240,160))
    tela.blit(texto_menu_2, (240,200))
    tela.blit(texto_menu_3, (240,240))

    global tempo_frame
    global frame_atual
    global animacao_terminou
    # desenha a porta na frente
    tela.blit(
        frames_cadeia[frame_atual],
        onde_frames_cadeia
    )
    # atualiza animação
    if not animacao_terminou:
        tempo_frame += 1
        if tempo_frame >= velocidade_animacao:
            tempo_frame = 0
            if frame_atual < len(frames_cadeia)-1:
                frame_atual += 1
            else:
                animacao_terminou = True

# Jornal
jornal_intro = pg.image.load(
    "FINAL_PCD_IMAGENS/Captura de tela 2026-05-11 105120.png"
).convert_alpha()

jornal_intro = pg.transform.smoothscale(
    jornal_intro,
    (546,585)
)

onde_jornal_intro = (
    (largura-546)/2,
    (altura-585)/2
)



def desenhar_jornal():

    tela.fill(cor_do_fundo)
    # desenha o jornal atrás
    tela.blit(
        jornal_intro,
        onde_jornal_intro
    )


def desenhar_tela_instrucoes():

    tela.fill(cor_do_fundo)

    # INSTRUÇÕES --------------------------------------------------
    texto_tela_instrucoes_1 = arial_huge.render(
        "INSTRUÇÕES:", 
        True, 
        branco
    )
    texto_tela_instrucoes_2 = arial_pequeno.render(
        "Você, como um prisioneiro sob custódia, tem um objetivo: tentar achar",
        True,
        branco
    )
    texto_tela_instrucoes_3 = arial_pequeno.render(
        "qual é a estratégia ótima - TRAIR ou COOPERAR - para cada situação",
        True,
        branco
    )
    texto_tela_instrucoes_4 = arial_pequeno.render(
        "(lembre-se de que estratégia ótima é aquela que resultará na pena total",
        True,
        branco
    )
    texto_tela_instrucoes_5 = arial_pequeno.render(
        "mínima). Aqui está a tabela explicando as penas:",
        True,
        branco
    )
    tela.blit(texto_tela_instrucoes_1, (50,70))
    tela.blit(texto_tela_instrucoes_2, (50,130))
    tela.blit(texto_tela_instrucoes_3, (50,160))
    tela.blit(texto_tela_instrucoes_4, (50,190))
    tela.blit(texto_tela_instrucoes_5, (50,220))
    
    def desenhar_tabela():

        tabela_instrucoes = pg.Rect(40, 280, 600, 240)
        pg.draw.rect(tela, cinza, tabela_instrucoes)

        x = 40
        y = 280

        largura_tabela = 200
        altura_tabela = 80

        # Desenha as linhas horizontais
        for i in range(4):
            pg.draw.line(
                tela,
                bege,
                (x, y + i*altura_tabela),
                (x + 3*largura_tabela, y + i*altura_tabela),
                1
            )
        # Desenha as linhas verticais
        for i in range(4):
            pg.draw.line(
                tela,
                bege,
                (x + i*largura_tabela, y),
                (x + i*largura_tabela, y + 3*altura_tabela),
                1
            )
        # Textos da tabela
        textos = [
            [" PLAYER 1 | PLAYER 2  ", "COOPERAR", "TRAIR"],
            ["COOPERAR", "2 anos | 2 anos", "10 anos | 1 ano"],
            ["TRAIR", "1 ano | 10 anos", "5 anos | 5 anos"]
        ]

        # Escreve os textos centralizados em cada célula
        for linha in range(3):
            for coluna in range(3):

                texto = arial_pequeno.render(
                    textos[linha][coluna],
                    True,
                    branco
                )

                centro_x = x + coluna*largura_tabela + largura_tabela/2
                centro_y = y + linha*altura_tabela + altura_tabela/2

                pos = texto.get_rect(center=(centro_x, centro_y))

                tela.blit(texto, pos)

        
        
    desenhar_tabela()

    

def desenhar_tela_fases():

    tela.fill(cor_do_fundo)

    # EXPLICANDO
    texto_tela_fases_1 = arial_huge.render(
        "SITUAÇÕES:", 
        True, 
        branco
    )
    texto_tela_fases_2 = arial_pequeno.render(
        "Em cada uma das situações, você vai encontrar um 'bot' diferente, ca-",
        True,
        branco
    )
    texto_tela_fases_3 = arial_pequeno.render(
        "da qual com um comportamento distinto (exemplo: um sempre trai, outro",
        True,
        branco
    )
    texto_tela_fases_4 = arial_pequeno.render(
        "escolhe aleatoriamente, etc). Aqui embaixo, você verá que as fases estão",
        True,
        branco
    )
    texto_tela_fases_5 = arial_pequeno.render(
        "divididas em dois grupos. Você entenderá isso depois...",
        True,
        branco
    )
    
    tela.blit(texto_tela_fases_1, (50,70))
    tela.blit(texto_tela_fases_2, (50,130))
    tela.blit(texto_tela_fases_3, (50,160))
    tela.blit(texto_tela_fases_4, (50,190))
    tela.blit(texto_tela_fases_5, (50,220))

    # DIFERENTES ETAPAS ------------------------------------
    texto_etapa_nao_lembra = arial.render(
        "SEM MEMÓRIA:",
        True,
        branco
    )

    texto_etapa_lembra = arial.render(
        "COM MEMÓRIA:",
        True,
        branco
    )

    pg.draw.rect(tela, cinza, etapa_nao_lembra)
    tela.blit(texto_etapa_nao_lembra, (86, 290))
    pg.draw.rect(tela, bege, bot_random)
    pg.draw.rect(tela, bege, bot_bom)
    pg.draw.rect(tela, bege, bot_mal)

    pg.draw.rect(tela, cinza, etapa_lembra)
    tela.blit(texto_etapa_lembra, (386, 290))
    pg.draw.rect(tela, bege, bot_vingativo)
    pg.draw.rect(tela, bege, bot_imitador)
    pg.draw.rect(tela, bege, bot_bravo)

    nomes_bots = [
        ("RANDOM", bot_random),
        ("BONZINHO", bot_bom),
        ("MALVADÃO", bot_mal),
        ("VINGATIVO", bot_vingativo),
        ("IMITADOR", bot_imitador),
        ("BRAVO", bot_bravo)
    ]

    for nome, botao in nomes_bots:


        texto = arial_pequeno.render(
            nome,
            True,
            preto
        )

        pos_texto = texto.get_rect(center=botao.center)

        tela.blit(texto, pos_texto)
 

def desenhar_tela_random(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):


    texto_tela_random_1 = arial_huge.render("BOT RANDOM", True, branco)

    texto_tela_random_2 = arial_pequeno.render(
        "Ele joga aleatoriamente",
        True,
        branco
    )

    tela.fill(cor_do_fundo)

    tela.blit(texto_tela_random_1, (50,60))
    tela.blit(texto_tela_random_2, (50,110))

    desenhar_escolhas(
        machine_choice, 
        user_choice, 
        mouse_pos, 
        pode_clicar,
        botao_cooperar,
        botao_trair,
        total_points,
        T, 
        C
    )




def desenhar_tela_bonzinho(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):
    tela.fill(cor_do_fundo)

    desenhar_escolhas(
        machine_choice, 
        user_choice, 
        mouse_pos, 
        pode_clicar,
        botao_cooperar,
        botao_trair,
        total_points,
        T, 
        C
    )

    texto_tela_bonzinho_1 = arial_huge.render("BOT BONZINHO", True, branco)
    texto_tela_bonzinho_2 = arial_pequeno.render(
        "Ele sempre irá cooperar",
        True,
        branco
    )
    tela.blit(texto_tela_bonzinho_1, (50,60))
    tela.blit(texto_tela_bonzinho_2, (50,110))

def desenhar_tela_malvado(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):
    tela.fill(cor_do_fundo)

    desenhar_escolhas(
        machine_choice, 
        user_choice, 
        mouse_pos, 
        pode_clicar,
        botao_cooperar,
        botao_trair,
        total_points,
        T, 
        C
    )

    texto_tela_malvado_1 = arial_huge.render("BOT MALVADÃO", True, branco)
    texto_tela_malvado_2 = arial_pequeno.render(
        "Ele sempre irá trair",
        True,
        branco
    )
    tela.blit(texto_tela_malvado_1, (50,60))
    tela.blit(texto_tela_malvado_2, (50,110))


def desenhar_tela_vingativo(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):
    tela.fill(cor_do_fundo)

    desenhar_escolhas(
        machine_choice, 
        user_choice, 
        mouse_pos, 
        pode_clicar,
        botao_cooperar,
        botao_trair,
        total_points,
        T, 
        C
    )

    texto_tela_vingativo_1 = arial_huge.render("BOT VINGATIVO", True, branco)
    texto_tela_vingativo_2 = arial_pequeno.render(
        "Ele é bonzinho, mas se você trair ele só uma vez, ele fica malvadão",
        True,
        branco
    )
    tela.blit(texto_tela_vingativo_1, (50,60))
    tela.blit(texto_tela_vingativo_2, (50,110))

def desenhar_tela_imitador(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):
    tela.fill(cor_do_fundo)

    desenhar_escolhas(
        machine_choice, 
        user_choice, 
        mouse_pos, 
        pode_clicar,
        botao_cooperar,
        botao_trair,
        total_points,
        T, 
        C
    )

    texto_tela_imitador_1 = arial_huge.render("BOT IMITADOR", True, branco)
    texto_tela_imitador_2 = arial_pequeno.render(
        "Ele imita o que você jogou na última rodada",
        True,
        branco
    )
    tela.blit(texto_tela_imitador_1, (50,60))
    tela.blit(texto_tela_imitador_2, (50,110))

def desenhar_tela_bravo(
    machine_choice, 
    user_choice, 
    mouse_pos, 
    pode_clicar,
    botao_cooperar,
    botao_trair,
    total_points,
    T, 
    C
):
    tela.fill(cor_do_fundo)

    desenhar_escolhas(
        machine_choice, 
        user_choice, 
        mouse_pos, 
        pode_clicar,
        botao_cooperar,
        botao_trair,
        total_points,
        T, 
        C
    )

    texto_tela_bravo_1 = arial_huge.render("BOT BRAVO", True, branco)
    texto_tela_bravo_2 = arial_pequeno.render(
        "Se você trair ele, ele retalia traindo duas vezes seguidas.",
        True,
        branco
    )
    tela.blit(texto_tela_bravo_1, (50,60))
    tela.blit(texto_tela_bravo_2, (50,110))

# =========================================================
# VARIÁVEIS DO JOGO
# =========================================================

estado = "menu"
rodando = True
pode_clicar = True
user_choice = None
machine_choice = None
resultado = 0
total_points = 0
C = 0 # cooperações totais
T = 0 # traições totais
c = 0 # cooperações seguidas
t = 0 # traições seguidas

pode_clicar = True
user_choice = None
machine_choice = None
resultado = 0
total_points = 0
C = 0
T = 0

# =========================================================
# RODANDO
# =========================================================

while rodando:

    mouse_pos = pg.mouse.get_pos() #pega a posição do mouse

    for event in pg.event.get():
        if event.type == pg.QUIT: # fecha o jogo se você fechar a janela
            pg.quit()
            sys.exit()
    
        elif event.type == pg.KEYDOWN: #se o evento for uma tecla apertada
            resetar()
            if event.key == pg.K_SPACE: # fecha o jogo
                rodando = False
            elif event.key == pg.K_r: # deixa você jogar de novo
                resetar()
                
            # MENU
            if estado == "menu":
                if event.key == pg.K_RIGHT:
                    estado = "jornal"

            # TELA JORNAL
            elif estado == "jornal":
                if event.key == pg.K_RIGHT:
                    estado = "tela_instrucoes"
                elif event.key == pg.K_LEFT:
                    estado = "menu"

            # TELA INSTRUÇÕES
            elif estado == "tela_instrucoes":
                if event.key == pg.K_LEFT:
                    estado = "jornal"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_fases"

            # TELA FASES
            elif estado == "tela_fases":
                if event.key == pg.K_LEFT:
                    estado = "tela_instrucoes"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_random"

            # TELA RANDOM
            elif estado == "tela_random":
                if event.key == pg.K_LEFT:
                    estado = "tela_instrucoes"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_bonzinho"

            # TELA BONZINHO
            elif estado == "tela_bonzinho":
                if event.key == pg.K_LEFT:
                    estado = "tela_random"
                elif event.key == pg.K_RIGHT:
                    estado = "tela_malvado"

            # TELA MALVADO
            elif estado == "tela_malvado":
                if event.key == pg.K_LEFT:
                    estado = "tela_bonzinho"
                if event.key == pg.K_RIGHT:
                    estado = "tela_vingativo"

            # TELA VINGATIVO
            elif estado == "tela_vingativo":
                if event.key == pg.K_LEFT:
                    estado = "tela_bonzinho"
                if event.key == pg.K_RIGHT:
                    estado = "tela_imitador"
            
            # TELA IMITADOR
            elif estado == "tela_imitador":
                if event.key == pg.K_LEFT:
                    estado = "tela_vingativo"
                if event.key == pg.K_RIGHT:
                    estado = "tela_bravo"

            # TELA BRAVO
            elif estado == "tela_bravo":
                if event.key == pg.K_LEFT:
                    estado = "tela_imitador"
                if event.key == pg.K_RIGHT:
                    estado = "tela_instrucoes"


        if event.type == pg.MOUSEBUTTONDOWN: #se o evento for um clique do mouse
                if event.button == 1 and pode_clicar:

                    if estado == "tela_fases":
                        if bot_random.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_random"
                            continue
                        if bot_bom.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_bonzinho"
                            continue
                        if bot_mal.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_malvado"
                            continue
                        if bot_vingativo.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_vingativo"
                            continue
                        if bot_imitador.collidepoint(mouse_pos):
                            resetar()
                            estado = "tela_imitador"
                            continue


                    if estado == "tela_random": # BOT RANDOM
                        # COOPERAR --------------------------------------------
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = randomico()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1
                            t = 0
                            c += 1
                            pode_clicar = False
                        # TRAIR -----------------------------------------------
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = randomico()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1
                            t += 1
                            c = 0
                            pode_clicar = False
            
                    if estado == "tela_bonzinho": # BOT BONZINHO
                        # COOPERAR --------------------------------------------
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = bonzinho()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1
                            t = 0
                            c += 1
                            pode_clicar = False
                        # TRAIR -----------------------------------------------
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = bonzinho()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1
                            t += 1
                            c = 0
                            pode_clicar = False

                    if estado == "tela_malvado": # BOT MALVADO
                        # COOPERAR --------------------------------------------
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = malvado()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1
                            t = 0
                            c += 1
                            pode_clicar = False
                        # TRAIR -----------------------------------------------
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = malvado()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1
                            t += 1
                            c = 0
                            pode_clicar = False

                    if estado == "tela_vingativo": # BOT VINGATIVO
                        # COOPERAR --------------------------------------------
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = vingativo()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1
                            t = 0
                            c += 1
                            pode_clicar = False
                        # TRAIR -----------------------------------------------
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = vingativo()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1
                            t += 1
                            c = 0
                            pode_clicar = False
        
                    if estado == "tela_imitador": # BOT IMITADOR
                        # COOPERAR --------------------------------------------
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = imitador()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1
                            t = 0
                            c += 1
                            pode_clicar = False
                        # TRAIR -----------------------------------------------
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = imitador()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1
                            t += 1
                            c = 0
                            pode_clicar = False

                    if estado == "tela_bravo": # BOT BRAVO
                        # COOPERAR --------------------------------------------
                        if botao_cooperar.collidepoint(mouse_pos):
                            user_choice = "cooperar"
                            machine_choice = bravo()
                            resultado = payoff(machine_choice,user_choice)
                            C += 1
                            t = 0
                            c += 1
                            pode_clicar = False
                        # TRAIR -----------------------------------------------
                        elif botao_trair.collidepoint(mouse_pos):
                            user_choice = "trair"
                            machine_choice = bravo()
                            resultado = payoff(machine_choice,user_choice)
                            T += 1
                            t += 1
                            c = 0
                            pode_clicar = False


    # =====================================================
    # DESENHO DAS TELAS
    # =====================================================

    if estado == "menu":

        desenhar_menu()

    elif estado == "jornal":

        desenhar_jornal()
    
    elif estado == "tela_instrucoes":

        desenhar_tela_instrucoes()
    
    elif estado == "tela_fases":
        
        desenhar_tela_fases()

    elif estado == "tela_random":

        desenhar_tela_random(
            machine_choice, 
            user_choice, 
            mouse_pos, 
            pode_clicar,
            botao_cooperar,
            botao_trair,
            total_points,
            T, 
            C
        )

        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    
    elif estado == "tela_bonzinho":

        desenhar_tela_bonzinho(
            machine_choice, 
            user_choice, 
            mouse_pos, 
            pode_clicar,
            botao_cooperar,
            botao_trair,
            total_points,
            T, 
            C
        )

        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True

    elif estado == "tela_malvado":

        desenhar_tela_malvado(
            machine_choice, 
            user_choice, 
            mouse_pos, 
            pode_clicar,
            botao_cooperar,
            botao_trair,
            total_points,
            T, 
            C
        )

        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
    
    elif estado == "tela_vingativo":

        desenhar_tela_vingativo(
            machine_choice, 
            user_choice, 
            mouse_pos, 
            pode_clicar,
            botao_cooperar,
            botao_trair,
            total_points,
            T, 
            C
        )

        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
            # user_choice = None
            # machine_choice = None
            # resultado = 0
            # print(total_points)
    
    elif estado == "tela_imitador":

        desenhar_tela_imitador(
            machine_choice, 
            user_choice, 
            mouse_pos, 
            pode_clicar,
            botao_cooperar,
            botao_trair,
            total_points,
            T, 
            C
        )

        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
            # user_choice = None
            # machine_choice = None
            # resultado = 0
            # print(total_points)
    

    elif estado == "tela_bravo":

        desenhar_tela_bravo(
            machine_choice, 
            user_choice, 
            mouse_pos, 
            pode_clicar,
            botao_cooperar,
            botao_trair,
            total_points,
            T, 
            C
        )

        if not pode_clicar:
            pg.display.flip()
            pg.time.wait(1000)
            total_points += resultado
            pode_clicar = True
            

    pg.display.flip()

pg.quit()
sys.exit() 

SystemExit: 